# 04 — Export Models to ONNX
**Indoor WiFi Fingerprinting Localization**

Converts all trained sklearn models to `.onnx` format for use in the Android Demo app.

**Key design choice:** all ONNX models accept **raw (unscaled) RSSI** as input.  
For kNN and SVM (which were trained on scaled data), the `StandardScaler` is baked  
into the pipeline before conversion — so the app always passes the same raw input.

### Output
```
Data/02_Processed_Wifi_Daytime/models/
  kNN.onnx
  Random_Forest.onnx
  SVM.onnx
  Gradient_Boosting.onnx

Demo/app/src/main/assets/models/     ← copied here for Android Studio
  kNN.onnx
  Random_Forest.onnx
  SVM.onnx
  Gradient_Boosting.onnx
  models_registry.json
  label_map.json
  feature_bssids.json
```

In [2]:
!pip install skl2onnx onnxruntime --quiet

In [3]:
import numpy as np
import json, joblib, shutil
from pathlib import Path

import skl2onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from sklearn.pipeline import Pipeline
import onnxruntime as rt

print(f'skl2onnx  : {skl2onnx.__version__}')
print(f'onnxruntime: {rt.__version__}')

skl2onnx  : 1.20.0
onnxruntime: 1.24.3


In [4]:
ROOT       = Path('../../')
SPLITS     = ROOT / 'Data' / '02_Processed_Wifi_Daytime' / 'splits'
MODELS     = ROOT / 'Data' / '02_Processed_Wifi_Daytime' / 'models'
ASSETS     = ROOT / 'Demo' / 'app' / 'src' / 'main' / 'assets' / 'models'
ASSETS.mkdir(parents=True, exist_ok=True)

print(f'Models dir : {MODELS.resolve()}')
print(f'Assets dir : {ASSETS.resolve()}')

Models dir : D:\Source_Codes\01_Ongoing\03_Indoor_Localization\Android\Data\02_Processed_Wifi_Daytime\models
Assets dir : D:\Source_Codes\01_Ongoing\03_Indoor_Localization\Android\Demo\app\src\main\assets\models


## 1. Load test data & supporting files

In [5]:
X_test         = np.load(SPLITS / 'test_X.npy').astype(np.float32)
y_test         = np.load(SPLITS / 'test_y.npy')
feature_bssids = json.load(open(SPLITS / 'feature_bssids.json'))
label_map      = json.load(open(SPLITS / 'label_map.json'))
registry       = json.load(open(MODELS / 'models_registry.json'))

N_FEATURES = len(feature_bssids)
print(f'Test samples : {len(X_test)}')
print(f'Features     : {N_FEATURES}')
print(f'Classes      : {len(label_map)}')
print(f'Models found : {list(registry.keys())}')

Test samples : 69
Features     : 116
Classes      : 19
Models found : ['kNN', 'Random Forest', 'SVM', 'Gradient Boosting']


## 2. Conversion helpers

In [6]:
def build_estimator(bundle):
    """
    Return the estimator to convert.
    For models that need scaling, wrap the pre-fit scaler + model in a Pipeline
    so the ONNX graph accepts raw RSSI input directly.
    """
    if bundle['needs_scaling']:
        return Pipeline([
            ('scaler', bundle['scaler']),
            ('clf',    bundle['model'])
        ])
    return bundle['model']


def to_onnx(estimator, n_features, model_name):
    """Convert a fitted sklearn estimator (or Pipeline) to ONNX bytes."""
    initial_type = [('rssi_input', FloatTensorType([None, n_features]))]
    return convert_sklearn(
        estimator,
        initial_types=initial_type,
        target_opset=17,
        options={id(estimator): {'zipmap': False}}  # output plain int labels, not dicts
        if not isinstance(estimator, Pipeline)
        else {id(estimator.steps[-1][1]): {'zipmap': False}}
    )


def verify_onnx(onnx_model_bytes, X, y_sklearn):
    """Run ONNX inference and compare labels against sklearn predictions."""
    sess   = rt.InferenceSession(onnx_model_bytes.SerializeToString())
    inp    = sess.get_inputs()[0].name
    labels = sess.run(None, {inp: X})[0]        # first output = predicted labels
    match  = np.sum(labels == y_sklearn)
    return labels, match / len(y_sklearn)


print('Helpers ready.')

Helpers ready.


## 3. Convert, verify and save each model

In [7]:
onnx_registry = {}   # updated registry that includes onnx file info

for model_name, info in registry.items():
    safe_name = model_name.replace(' ', '_')
    onnx_filename = f'{safe_name}.onnx'

    # ── Load bundle ──────────────────────────────────────────────────────────
    bundle    = joblib.load(MODELS / info['file'])
    estimator = build_estimator(bundle)

    # ── Convert ──────────────────────────────────────────────────────────────
    onnx_model = to_onnx(estimator, N_FEATURES, model_name)

    # ── Verify: ONNX labels must match sklearn labels on test set ─────────────
    sklearn_preds = bundle['model'].predict(
        bundle['scaler'].transform(X_test) if bundle['needs_scaling'] else X_test
    )
    onnx_preds, agreement = verify_onnx(onnx_model, X_test, sklearn_preds)

    if agreement < 1.0:
        print(f'  WARNING  {model_name}: ONNX/sklearn agreement = {agreement:.1%}')
    
    # ── Save to models dir ───────────────────────────────────────────────────
    models_path = MODELS / onnx_filename
    with open(models_path, 'wb') as f:
        f.write(onnx_model.SerializeToString())

    # ── Copy to Android assets ───────────────────────────────────────────────
    assets_path = ASSETS / onnx_filename
    shutil.copy2(models_path, assets_path)

    size_kb = models_path.stat().st_size / 1024
    print(f'  OK  {model_name:<25}  agreement={agreement:.1%}  size={size_kb:.1f} KB')
    print(f'      → {models_path.name}')
    print(f'      → {assets_path}')

    onnx_registry[model_name] = {
        **info,
        'onnx_file'   : onnx_filename,
        'size_kb'     : round(size_kb, 1),
        'needs_scaling': False,   # scaler is baked in — app always passes raw RSSI
    }
    print()

  WARNING  kNN: ONNX/sklearn agreement = 95.7%
  OK  kNN                        agreement=95.7%  size=184.9 KB
      → kNN.onnx
      → ..\..\Demo\app\src\main\assets\models\kNN.onnx

  OK  Random Forest              agreement=100.0%  size=2172.5 KB
      → Random_Forest.onnx
      → ..\..\Demo\app\src\main\assets\models\Random_Forest.onnx

  OK  SVM                        agreement=100.0%  size=288.8 KB
      → SVM.onnx
      → ..\..\Demo\app\src\main\assets\models\SVM.onnx

  OK  Gradient Boosting          agreement=100.0%  size=2539.2 KB
      → Gradient_Boosting.onnx
      → ..\..\Demo\app\src\main\assets\models\Gradient_Boosting.onnx



## 4. Copy supporting JSON files to assets

In [8]:
# Updated registry (onnx_file field added, needs_scaling=False for all)
with open(ASSETS / 'models_registry.json', 'w') as f:
    json.dump(onnx_registry, f, indent=2)

# label_map and feature_bssids — the app needs both at runtime
shutil.copy2(SPLITS / 'label_map.json',       ASSETS / 'label_map.json')
shutil.copy2(SPLITS / 'feature_bssids.json',  ASSETS / 'feature_bssids.json')

print('Copied to assets:')
for f in sorted(ASSETS.iterdir()):
    print(f'  {f.name:<35}  {f.stat().st_size/1024:.1f} KB')

Copied to assets:
  feature_bssids.json                  2.7 KB
  Gradient_Boosting.onnx               2539.2 KB
  kNN.onnx                             184.9 KB
  label_map.json                       0.6 KB
  models_registry.json                 0.8 KB
  Random_Forest.onnx                   2172.5 KB
  SVM.onnx                             288.8 KB


## 5. Sanity-check: end-to-end inference via ONNX
Simulates exactly what the Android app will do:  
`{bssid: rssi}` scan dict → feature vector → ONNX inference → location name.

In [9]:
def predict_onnx(scan_dict: dict, onnx_path: Path,
                 feature_bssids: list, label_map: dict,
                 rssi_missing: float = -100.0) -> tuple[str, float]:
    """
    Predict location from a raw WiFi scan using an ONNX model.
    Returns (location_name, confidence_if_available).
    This is the reference implementation for the Android port.
    """
    # 1. Build feature vector (same order as training)
    x = np.array(
        [scan_dict.get(bssid, rssi_missing) for bssid in feature_bssids],
        dtype=np.float32
    ).reshape(1, -1)

    # 2. Run ONNX session
    sess    = rt.InferenceSession(str(onnx_path))
    inp     = sess.get_inputs()[0].name
    outputs = sess.run(None, {inp: x})

    # outputs[0] = label (int), outputs[1] = probabilities (if available)
    pred_idx = int(outputs[0][0])
    location = label_map[str(pred_idx)]

    confidence = None
    if len(outputs) > 1:
        probs = outputs[1][0]   # shape: (n_classes,)
        confidence = float(probs[pred_idx])

    return location, confidence


# Use test sample #0 as a mock phone scan
sample_raw  = X_test[0]
scan_dict   = {bssid: float(rssi)
               for bssid, rssi in zip(feature_bssids, sample_raw)
               if rssi > -100}
actual      = label_map[str(int(y_test[0]))]

print(f'APs in scan : {len(scan_dict)} / {N_FEATURES} total')
print(f'Actual      : {actual}')
print()

for model_name, info in sorted(onnx_registry.items(),
                                key=lambda x: -x[1]['test_accuracy']):
    location, conf = predict_onnx(
        scan_dict, ASSETS / info['onnx_file'], feature_bssids, label_map
    )
    ok   = '✓' if location == actual else '✗'
    conf_str = f'  conf={conf:.0%}' if conf is not None else ''
    print(f'  {ok}  {model_name:<25}  → {location}{conf_str}')

APs in scan : 40 / 116 total
Actual      : Btw_Stairs_and_DLT7

  ✓  Random Forest              → Btw_Stairs_and_DLT7  conf=53%
  ✗  SVM                        → Near_Stairs_To_1st_Floor  conf=1832%
  ✓  Gradient Boosting          → Btw_Stairs_and_DLT7  conf=79%
  ✓  kNN                        → Btw_Stairs_and_DLT7  conf=67%


## 6. Summary

In [10]:
print('=' * 60)
print('ONNX EXPORT SUMMARY')
print('=' * 60)
print(f'  Input  : float32[1, {N_FEATURES}]  (raw RSSI, missing=-100)')
print(f'  Output : int64 label  (use label_map.json to decode)')
print()
print(f'  {"Model":<25} {"Test Acc":>10} {"Size":>10}')
print(f'  {"-"*25} {"-"*10} {"-"*10}')
for name, info in sorted(onnx_registry.items(), key=lambda x: -x[1]['test_accuracy']):
    print(f'  {name:<25} {info["test_accuracy"]:>9.1%} {info["size_kb"]:>9.1f} KB')
print()
print(f'  Assets → {ASSETS.resolve()}')
print()
print('Android Gradle dependency:')
print("  implementation 'com.microsoft.onnxruntime:onnxruntime-android:1.20.0'")

ONNX EXPORT SUMMARY
  Input  : float32[1, 116]  (raw RSSI, missing=-100)
  Output : int64 label  (use label_map.json to decode)

  Model                       Test Acc       Size
  ------------------------- ---------- ----------
  Random Forest                 94.2%    2172.5 KB
  SVM                           88.4%     288.8 KB
  Gradient Boosting             88.4%    2539.2 KB
  kNN                           85.5%     184.9 KB

  Assets → D:\Source_Codes\01_Ongoing\03_Indoor_Localization\Android\Demo\app\src\main\assets\models

Android Gradle dependency:
  implementation 'com.microsoft.onnxruntime:onnxruntime-android:1.20.0'
